In [ ]:
!pip install pandas scikit-learn matplotlib seaborn plotly dash dash_core_components dash_html_components folium missingno

Importation des librairies nécessaires et de l'affichage

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import tarfile
import os

# Configuring display settings
plt.rcParams['figure.figsize'] = (12, 9)
sns.set()
sns.set_context('talk')
np.set_printoptions(threshold=20, precision=2, suppress=True)
pd.set_option('display.max_rows', 30)
pd.set_option('display.max_columns', None)
pd.set_option('display.precision', 2)
pd.set_option('display.float_format', '{:.2f}'.format)
warnings.filterwarnings("ignore", category=FutureWarning)

Nous utiliserons les données immobilières californiennes de Kaggle. Nous exploiterons les données de localisation (latitude et longitude) ainsi que la valeur médiane des maisons. Nous regrouperons les maisons par emplacement et observerons les fluctuations des prix immobiliers en Californie.

In [ ]:
import pandas as pd

housing_data = pd.read_csv('../housing_data/housing.csv', usecols = ['longitude', 'latitude', 'median_house_value'])
housing_data.head()
housing_data.info()
housing_data.describe()

Gérance des données manquantes

In [ ]:
import missingno as msno
import seaborn as sns
import matplotlib.pyplot as plt

# Visualize missing data
msno.matrix(housing_data)
plt.show()

# Impute missing values
# For numerical columns, use median imputation
numerical_columns = housing_data.select_dtypes(include=['float64', 'int64']).columns
housing_data[numerical_columns] = housing_data[numerical_columns].fillna(housing_data[numerical_columns].median())

# For categorical columns, use mode imputation
# Sélection des colonnes catégorielles
categorical_columns = housing_data.select_dtypes(include=['object']).columns

# On boucle sur chaque colonne pour éviter l'erreur si une colonne est vide
for col in categorical_columns:
    mode_series = housing_data[col].mode()
    if not mode_series.empty:
        housing_data[col] = housing_data[col].fillna(mode_series[0])
    else:
        # Optionnel : si la colonne est 100% vide, on peut mettre une valeur par défaut
        housing_data[col] = housing_data[col].fillna("Unknown")

# Vérification
print(housing_data[categorical_columns].isnull().sum())

# Verify that there are no more missing values
housing_data.isnull().sum()

Nous commençons par visualiser nos données immobilières. Nous analysons les données de localisation à l'aide d'une carte thermique basée sur le prix médian par îlot. Dans ce tutoriel, nous utiliserons Seaborn pour créer rapidement des graphiques

In [ ]:
import seaborn as sns

sns.scatterplot(data = housing_data, x = 'longitude', y = 'latitude', hue = 'median_house_value')

Lorsqu'on utilise des algorithmes basés sur la distance, comme le clustering k-means, il est indispensable de normaliser les données. Sans normalisation, les variables d'échelles différentes seront pondérées différemment dans la formule de distance optimisée lors de l'entraînement. Par exemple, si l'on incluait le prix dans le cluster, en plus de la latitude et de la longitude, le prix aurait un impact disproportionné sur l'optimisation, car son échelle est nettement plus grande et plus étendue que celle des variables de localisation.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(housing_data[['latitude', 'longitude']], housing_data[['median_house_value']], test_size=0.33, random_state=0)

Ensuite, nous normalisons les données d'entraînement et de test en utilisant la preprocessing.normalize()méthode de sklearn. 

In [ ]:
from sklearn import preprocessing

X_train_norm = preprocessing.normalize(X_train)
X_test_norm = preprocessing.normalize(X_test)

Pour la première itération, nous choisirons arbitrairement 2 clusters (noté k). La construction et l'ajustement des modèles sklearnsont très simples. Nous créerons une instance de KMeans, définirons le nombre de clusters à l'aide de l' n_clustersattribut , définirons n_init, qui définit le nombre d'itérations que l'algorithme effectuera avec différentes valeurs initiales des centroïdes, sur « auto », et nous définirons random_stateà 0 afin d'obtenir le même résultat à chaque exécution du code. Nous pourrons ensuite ajuster le modèle aux données d'entraînement normalisées à l'aide de la fit()méthode .

In [ ]:
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters = 2, random_state = 0, n_init='auto')
kmeans.fit(X_train_norm)

In [ ]:
sns.scatterplot(data = X_train, x = 'longitude', y = 'latitude', hue = kmeans.labels_)

In [ ]:
On constate que les données se répartissent désormais clairement en 2 groupes. On peut également observer la distribution des prix médians des maisons dans ces trois groupes à l'aide d'un diagramme en boîte. 

In [ ]:
sns.boxplot(x = kmeans.labels_, y = y_train['median_house_value'])

Nous pouvons évaluer les performances de l'algorithme de clustering à l'aide d'un score Silhouette, où sklearn.metricsun score plus bas représente une meilleure adéquation

In [ ]:
from sklearn.metrics import silhouette_score

silhouette_score(X_train_norm, kmeans.labels_, metric='euclidean')

N'ayant pas étudié l'influence du nombre de clusters sur la qualité des résultats, nous ignorons la pertinence du modèle à k = 3. Dans la section suivante, nous explorerons différents nombres de clusters et comparerons leurs performances afin de déterminer les valeurs optimales des hyperparamètres pour notre modèle.

Le principal inconvénient du clustering k-means est qu'il est impossible de déterminer le nombre de clusters nécessaires par la simple exécution du modèle. Il est donc nécessaire de tester différentes valeurs de k afin de choisir la valeur optimale. On utilise généralement la méthode du coude pour déterminer le nombre optimal de clusters, évitant ainsi le surapprentissage (avec un nombre trop élevé de clusters) et le sous-apprentissage (avec un nombre trop faible de clusters). 

In [ ]:
K = range(2, 8)
fits = []
score = []


for k in K:
    # train the model for current value of k on training data
    model = KMeans(n_clusters = k, random_state = 0, n_init='auto').fit(X_train_norm)
    
    # append the model to fits
    fits.append(model)
    
    # Append the silhouette score to scores
    score.append(silhouette_score(X_train_norm, model.labels_, metric='euclidean'))

Nous examinons k=2

In [ ]:
sns.scatterplot(data = X_train, x = 'longitude', y = 'latitude', hue = fits[0].labels_)

k=4

In [ ]:
sns.scatterplot(data = X_train, x = 'longitude', y = 'latitude', hue = fits[2].labels_)

k=7

In [ ]:
sns.scatterplot(data = X_train, x = 'longitude', y = 'latitude', hue = fits[5].labels_)

En général, lorsque l'on augmente la valeur de K, on ​​observe une amélioration des clusters et de leur représentation jusqu'à un certain point. Au-delà, les gains diminuent, voire les performances se dégradent. On peut visualiser ce phénomène et faciliter le choix de la valeur de k à l'aide d'un graphique en coude, où l'axe des ordonnées représente la qualité de l'ajustement et l'axe des abscisses la valeur de k.

In [ ]:
sns.lineplot(x = K, y = score)

On peut également constater que les regroupements réussissent relativement bien à diviser la Californie en groupes distincts et que ces groupes correspondent relativement bien à différentes gammes de prix, comme on peut le voir ci-dessous. 

In [ ]:
sns.scatterplot(data = X_train, x = 'longitude', y = 'latitude', hue = fits[3].labels_)

In [ ]:
sns.boxplot(x = fits[3].labels_, y = y_train['median_house_value'])